In [1]:
import sys
sys.path.insert(0, '../')
sys.path.insert(1, '../../')
from build_config import ALL_SEASONS, COMPETITIONS, CURRENT_SEASON, INPUT_CSV_PATHS, TARGET_RANGES
from const import REPO_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, MODELS_PATH
from fbref_const import URLs, TARGET_COLUMNS

from src.data.match_data_processing import process_match_target_var, process_match_other_var
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature

import pandas as pd
import os
import numpy as np
import mlflow
import joblib
import tempfile
import os

from build_utils import *

In [2]:
key_columns=['date', 'home', 'away']

In [3]:
team_encoder_path = f"{MODELS_PATH}/data_processors/{COMPETITIONS[0]}/team_encoder.pkl"
encoder = TeamEncoder.load(team_encoder_path)

In [4]:
seasons=sorted(ALL_SEASONS)

In [5]:
test_dict=dict((k, pd.read_csv(f'{REPO_PATH}/{INPUT_CSV_PATHS[k]}')) for k in COMPETITIONS)

In [6]:
all_features_dict={}
for competition in COMPETITIONS:
    if competition not in test_dict:
        continue
    test_df = test_dict[competition]
    team_encoder_path = f"{MODELS_PATH}/data_processors/{competition}/team_encoder.pkl"
    season_dfs_dict = {season: pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons}
    target_dfs = [pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]
    # TeamEncoder features
    encoder = TeamEncoder.load(team_encoder_path)
    team_encoder_features = encoder.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagFeatureGenerator features
    lag_feature_generator = TeamLagFeatureGenerator(lookback=5)
    team_lag_features = lag_feature_generator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # PreviousSeasonTeamAverager features
    prev_season_averager = PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
    prev_season_features = prev_season_averager.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamRestDaysCalculator features
    rest_days_calculator = TeamRestDaysCalculator()
    rest_days_features = rest_days_calculator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagTargetFeature features
    lag_target_generator = TeamLagTargetFeature(lookback=5)
    lag_target_features = lag_target_generator.transform_spot(season_dfs_dict, target_dfs, test_df).reset_index(drop=True)

    # Merge all features on home, away, date
    from functools import reduce
    feature_dfs = [team_encoder_features, team_lag_features, prev_season_features, rest_days_features, lag_target_features]
    all_features = reduce(lambda left, right: pd.merge(left, right, on=['home', 'away', 'date'], how='outer'), feature_dfs)
    
    scaler = joblib.load(os.path.join(f"{MODELS_PATH}/data_processors/{competition}", 'standard_scaler.pkl'))
    all_features[scaler.feature_names_in_] = scaler.transform(all_features[scaler.feature_names_in_])

    all_features_dict[competition] = all_features.copy().fillna(-1)

In [7]:
train_features=pd.read_csv(f"{REPO_PATH}/data/features/premier_league/all_combined_features_2017-24.csv")
train_features['date'] = pd.to_datetime(train_features['date'])
train_features = train_features.merge(all_features_dict['premier_league'][['home', 'away', 'date']], on=['home', 'away', 'date'], how='right').fillna(-1)

for competition, df in all_features_dict.items():
    all_features_dict[competition] = df[train_features.columns]

In [8]:
# Compare train_features and all_features_dict['premier_league'] (excluding home, away, date)
tf_values = train_features.drop(columns=['home', 'away', 'date']).values
af_values = all_features_dict['premier_league'].drop(columns=['home', 'away', 'date']).values

# Find where they differ
diff_mask = 1-np.isclose(tf_values, af_values)
diff_indices = np.argwhere(diff_mask)
print(f"Number of differing values: {diff_indices.shape[0]}")
print("First 10 differences:")
train_feature_cols = train_features.drop(columns=['home', 'away', 'date']).columns.tolist()
all_feature_cols = all_features_dict['premier_league'].drop(columns=['home', 'away', 'date']).columns.tolist()
for idx, (row, col) in enumerate(diff_indices[:10]):
    home = train_features.iloc[row]['home']
    away = train_features.iloc[row]['away']
    date = train_features.iloc[row]['date']
    tf_val = tf_values[row, col]
    af_val = af_values[row, col]
    col_name = train_feature_cols[col]
    col_name_af = all_feature_cols[col]

    print(f"Row {row}, Column '{col_name}, {col_name_af}': home={home}, away={away}, date={date}, train_features={tf_val}, all_features_dict={af_val}")

Number of differing values: 0
First 10 differences:


In [ ]:
model_dict={
    'premier_league': {
        'away_goals':{
            'run_id': 'af35f46144664c1b8bf517853c2b66dd',
            'artifact_path': 'model',
        },
        'home_goals':{
            'run_id': '99f5aa45e9a14dbf96f05773aeac29a8',
            'artifact_path': 'model',
        },
    }
}

In [10]:
def load_joblib_model_from_mlflow(run_id, artifact_path, tracking_uri=None):
    """
    Fetch and load a joblib-dumped model from MLflow given experiment_id, run_id, and artifact_path.
    Optionally specify the MLflow tracking URI.
    Returns the loaded model.
    """
    if tracking_uri is not None:
        mlflow.set_tracking_uri(tracking_uri)
    client = mlflow.tracking.MlflowClient()
    # Download artifact to a temporary directory
    with tempfile.TemporaryDirectory() as tmp_dir:
        local_path = client.download_artifacts(run_id, artifact_path, tmp_dir)
        # Find the first .joblib file in the artifact directory
        for root, _, files in os.walk(local_path):
            for file in files:
                if file.endswith('.joblib'):
                    model_path = os.path.join(root, file)
                    return joblib.load(model_path)
        raise FileNotFoundError("No .joblib model file found in the artifact path.")


In [11]:
predictions={}

In [12]:
for competition in COMPETITIONS:
    predictions[competition] = {}
    for target_name in TARGET_RANGES:
        model = load_joblib_model_from_mlflow(model_dict[competition][target_name]['run_id'],
                                                model_dict[competition][target_name]['artifact_path'],
                                                f'{REPO_PATH}/mlflow')
        prediction = model.predict_proba(all_features_dict[competition].drop(columns=key_columns))
        prediction = pd.DataFrame(prediction, columns=list(range(TARGET_RANGES[target_name][0], TARGET_RANGES[target_name][1]+1))+['other'])
        predictions[competition][target_name]=prediction

/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
all_features_dict[competition]['home_lag1_home_pass_types_crs']

0   -0.067953
1   -1.713903
2    1.577997
3    2.871243
4    1.813132
5   -0.067953
6    1.577997
7   -1.000000
8   -1.713903
9    0.049615
Name: home_lag1_home_pass_types_crs, dtype: float64

In [13]:
# Add home, away, date columns from test_dict to each predictions DataFrame
for competition in predictions:
    test_rows = all_features_dict[competition][['home', 'away', 'date']].reset_index(drop=True)
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        # Prepend home, away, date columns
        df = pd.concat([test_rows, df.reset_index(drop=True)], axis=1)
        predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [14]:
predictions['premier_league']['home_goals']

,home,away,date,0,1,2,3,4,5,other
0,Aston Villa,Newcastle Utd,2025-08-16,0.167658,0.356361,0.255829,0.105283,0.066932,0.041825,0.006112
1,Brighton,Fulham,2025-08-16,0.145975,0.352849,0.160223,0.129326,0.113381,0.091612,0.006633
2,Chelsea,Crystal Palace,2025-08-17,0.245862,0.279064,0.224551,0.114570,0.071822,0.040962,0.023168
3,Leeds United,Everton,2025-08-18,0.400285,0.412621,0.093462,0.048962,0.029899,0.011691,0.003080
4,Liverpool,Bournemouth,2025-08-15,0.245375,0.353225,0.199073,0.117080,0.052687,0.023829,0.008732
5,Manchester Utd,Arsenal,2025-08-17,0.245688,0.329908,0.175040,0.147077,0.052066,0.044133,0.006088
6,Nott'ham Forest,Brentford,2025-08-17,0.288712,0.287006,0.188145,0.130035,0.062120,0.032728,0.011254
7,Sunderland,West Ham,2025-08-16,0.518608,0.303256,0.094360,0.043619,0.019600,0.017728,0.002830
8,Tottenham,Burnley,2025-08-16,0.134311,0.194776,0.326685,0.128733,0.116804,0.086714,0.011977
9,Wolves,Manchester City,2025-08-16,0.201177,0.244701,0.305567,0.115389,0.058310,0.065028,0.009828


In [15]:
predictions['premier_league']['away_goals']

,home,away,date,0,1,2,3,4,5,6,other
0,Aston Villa,Newcastle Utd,2025-08-16,0.158899,0.425739,0.159829,0.170977,0.065593,0.008407,0.009213,0.001343
1,Brighton,Fulham,2025-08-16,0.151249,0.396891,0.248957,0.126521,0.059944,0.006009,0.009425,0.001004
2,Chelsea,Crystal Palace,2025-08-17,0.208223,0.440585,0.239109,0.078263,0.025447,0.005100,0.002322,0.000951
3,Leeds United,Everton,2025-08-18,0.138559,0.215181,0.420033,0.166984,0.021784,0.028781,0.007158,0.001520
4,Liverpool,Bournemouth,2025-08-15,0.151334,0.378270,0.337780,0.097426,0.022615,0.006564,0.004917,0.001093
5,Manchester Utd,Arsenal,2025-08-17,0.198816,0.516421,0.204477,0.059673,0.012180,0.004442,0.003049,0.000942
6,Nott'ham Forest,Brentford,2025-08-17,0.232143,0.312459,0.287460,0.142767,0.012267,0.004408,0.005982,0.002513
7,Sunderland,West Ham,2025-08-16,0.113547,0.289410,0.422296,0.131013,0.013192,0.025178,0.004131,0.001233
8,Tottenham,Burnley,2025-08-16,0.349154,0.355668,0.173484,0.058651,0.057723,0.002242,0.001721,0.001357
9,Wolves,Manchester City,2025-08-16,0.224078,0.257493,0.369739,0.100483,0.033722,0.006240,0.007148,0.001096


In [16]:
# Rename columns and update values in predictions DataFrames for each competition and target
for competition in predictions:
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        cols = df.columns.tolist()
        # Only rename and update non-metadata columns (assume first three are home, away, date)
        meta_cols = ['home', 'away', 'date']
        feature_cols = cols[3:]
        # Rename columns: first becomes lte_{original}, others become gt_{previous}
        new_cols = meta_cols.copy()
        if feature_cols:
            new_cols.append(f"lte_{feature_cols[0]}")
            for i in range(1, len(feature_cols)):
                new_cols.append(f"gt_{feature_cols[i-1]}")
        # Update values: first feature column stays, others become sum of itself and all to the right
        arr = df[feature_cols].values.copy() if feature_cols else None
        if arr is not None and arr.shape[1] > 0:
            for i in range(1, arr.shape[1]):
                arr[:, i] = arr[:, i:].sum(axis=1)
            df = pd.concat([df[meta_cols].reset_index(drop=True), pd.DataFrame(arr, columns=new_cols[3:])], axis=1)
            df.columns = new_cols
            predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [17]:
predictions['premier_league']['home_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5
0,Aston Villa,Newcastle Utd,2025-08-16,0.167658,0.832342,0.475980,0.220151,0.114868,0.047936,0.006112
1,Brighton,Fulham,2025-08-16,0.145975,0.854025,0.501176,0.340953,0.211626,0.098245,0.006633
2,Chelsea,Crystal Palace,2025-08-17,0.245862,0.754138,0.475074,0.250522,0.135952,0.064130,0.023168
3,Leeds United,Everton,2025-08-18,0.400285,0.599715,0.187094,0.093632,0.044669,0.014771,0.003080
4,Liverpool,Bournemouth,2025-08-15,0.245375,0.754625,0.401400,0.202327,0.085247,0.032560,0.008732
5,Manchester Utd,Arsenal,2025-08-17,0.245688,0.754312,0.424404,0.249364,0.102287,0.050221,0.006088
6,Nott'ham Forest,Brentford,2025-08-17,0.288712,0.711288,0.424282,0.236137,0.106102,0.043983,0.011254
7,Sunderland,West Ham,2025-08-16,0.518608,0.481392,0.178136,0.083777,0.040158,0.020558,0.002830
8,Tottenham,Burnley,2025-08-16,0.134311,0.865689,0.670913,0.344228,0.215494,0.098691,0.011977
9,Wolves,Manchester City,2025-08-16,0.201177,0.798823,0.554121,0.248555,0.133166,0.074856,0.009828


In [18]:
predictions['premier_league']['away_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5,gt_6
0,Aston Villa,Newcastle Utd,2025-08-16,0.158899,0.841101,0.415362,0.255533,0.084556,0.018962,0.010555,0.001343
1,Brighton,Fulham,2025-08-16,0.151249,0.848751,0.451860,0.202903,0.076382,0.016438,0.010429,0.001004
2,Chelsea,Crystal Palace,2025-08-17,0.208223,0.791777,0.351192,0.112083,0.033820,0.008373,0.003273,0.000951
3,Leeds United,Everton,2025-08-18,0.138559,0.861441,0.646260,0.226227,0.059243,0.037459,0.008678,0.001520
4,Liverpool,Bournemouth,2025-08-15,0.151334,0.848666,0.470396,0.132617,0.035190,0.012575,0.006010,0.001093
5,Manchester Utd,Arsenal,2025-08-17,0.198816,0.801184,0.284763,0.080286,0.020613,0.008433,0.003991,0.000942
6,Nott'ham Forest,Brentford,2025-08-17,0.232143,0.767857,0.455398,0.167937,0.025170,0.012904,0.008496,0.002513
7,Sunderland,West Ham,2025-08-16,0.113547,0.886453,0.597043,0.174747,0.043734,0.030542,0.005364,0.001233
8,Tottenham,Burnley,2025-08-16,0.349154,0.650846,0.295179,0.121695,0.063043,0.005320,0.003079,0.001357
9,Wolves,Manchester City,2025-08-16,0.224078,0.775922,0.518429,0.148690,0.048206,0.014484,0.008244,0.001096
